In [86]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [87]:
df=pd.read_csv("train.txt",sep=';',header=None,names=['text','emotion'])

In [88]:
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [89]:
emotions=df['emotion'].unique()
emotions

<StringArray>
['sadness', 'anger', 'love', 'surprise', 'fear', 'joy']
Length: 6, dtype: str

In [90]:
emotion_number={}
i=0
for emo in emotions:
    emotion_number[emo]=i
    i+=1
df['emotion']=df['emotion'].map(emotion_number)

In [91]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [92]:
emotion_number

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}

In [93]:
df['text']=df['text'].apply(lambda x:x.lower())

In [94]:
# removing puntuations
import string

def remove_punct(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

df['text']=df['text'].apply(remove_punct)

In [95]:
# removing numbers
def remove_numbers(txt):
    new=""
    for word in txt:
        if not word.isdigit():
            new=new+word
    return new

df['text']=df['text'].apply(remove_numbers)


In [96]:
def remove_emoji(txt):
    new=""
    for i in txt:
        if i.isascii():
            new=new+i

    return new

df['text']=df['text'].apply(remove_emoji)

In [97]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [98]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [99]:
stop_words=set(stopwords.words('english'))

In [100]:
len(stop_words)

198

In [101]:
def remove_text(txt):
    words=word_tokenize(txt)
    cleaned=[]
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    
    return ' '.join(cleaned)

In [102]:
df['text']=df['text'].apply(remove_text)

In [103]:
# Using bag of words

In [105]:
from sklearn.feature_extraction.text import CountVectorizer
count_vec=CountVectorizer()
X=count_vec.fit_transform(df['text'])

In [107]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 144766 stored elements and shape (16000, 15044)>
  Coords	Values
  (0, 3585)	1
  (0, 4886)	1
  (0, 6370)	1
  (1, 5633)	1
  (1, 4890)	1
  (1, 6282)	1
  (1, 3175)	1
  (1, 6280)	1
  (1, 690)	1
  (1, 12273)	1
  (1, 1926)	1
  (1, 916)	1
  (2, 4886)	1
  (2, 6493)	1
  (2, 5688)	1
  (2, 8371)	1
  (2, 10048)	1
  (2, 5749)	1
  (2, 14899)	1
  (3, 4890)	1
  (3, 4525)	1
  (3, 8969)	1
  (3, 5015)	1
  (3, 7350)	1
  (3, 12620)	1
  :	:
  (15995, 1106)	1
  (15996, 4886)	1
  (15996, 12620)	1
  (15996, 9542)	1
  (15996, 13825)	1
  (15996, 13082)	1
  (15996, 3359)	1
  (15996, 14446)	1
  (15996, 13186)	1
  (15996, 12795)	1
  (15997, 4886)	1
  (15997, 5657)	1
  (15997, 12736)	1
  (15997, 9318)	1
  (15998, 4886)	1
  (15998, 6493)	1
  (15998, 7656)	1
  (15998, 5583)	1
  (15998, 11281)	1
  (15998, 2492)	1
  (15999, 4886)	1
  (15999, 7350)	1
  (15999, 7827)	1
  (15999, 12781)	1
  (15999, 10018)	1


In [ ]:
X[:5].toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

# Now doing prediction and checking accuracy

In [110]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [113]:
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# using naive baye model and multinomial nb because it is more suitable for text data 
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print("Bag of Words: ",accuracy_score(y_test, pred_bow))

Bag of Words:  0.7678125


## Now predicting with tfidfvectorize and checking which is working more better

In [114]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [115]:
tfidf_vectorizer=TfidfVectorizer()


In [116]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [117]:
y_pred = nb2_model.predict(X_test_tfidf)
print(accuracy_score(y_test,y_pred))

0.6609375


- both models are performing very bad on naivebaye . trying with logistic regression

In [118]:
from sklearn.linear_model import LogisticRegression

In [119]:
log_model=LogisticRegression(max_iter=1000)

In [121]:
log_model.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [122]:
log_pred=log_model.predict(X_test_tfidf)

In [124]:
print("Logistic Regression model",accuracy_score(y_test,log_pred))

Logistic Regression model 0.86125


## So here it is performing very well than others